# 03. Interactive Baseline Speech Generation Demo

**Course:** Advanced Machine Learning  
**Project:** Phonetic-Guided Speech Generation via Alignment & Latent Diffusion  
**Lead Evaluator:** Chaitanya Shekhawat (Member 5)

---

### Overview
This demo compares four speech synthesis systems on held-out test clips (`LJ005` prefix):
1. **R0:** Real recording (Original human voice)
2. **R1:** Copy synthesis (Real mel spectrogram $\to$ HiFi-GAN vocoder)
3. **A1:** 3-state phoneme templates + true ground-truth MFA durations
4. **A2:** 3-state phoneme templates + predicted durations (HistGradientBoosting)

In [ ]:
from pathlib import Path
import IPython.display as ipd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf

ROOT = Path("..").resolve()
SAMPLES_DIR = ROOT / "samples"
RESULTS_DIR = ROOT / "outputs" / "results"

print(f"Repo Root: {ROOT}")
print(f"Samples found: {len(list(SAMPLES_DIR.glob('*.wav')))} wav files")

## 1. Summary of Experimental Results
Objective evaluation scores computed over 100 held-out clips (`test_100`) using OpenAI Whisper (`small.en`).

In [ ]:
summary_df = pd.read_csv(RESULTS_DIR / "summary.csv")
summary_df

## 2. Audio Player & Spectrogram Comparison Function
Helper function to display audio players and mel-spectrogram visualizations side-by-side.

In [ ]:
def compare_clip(clip_id: str, label: str):
    print("=" * 70)
    print(f"{label}: {clip_id}")
    print("=" * 70)
    
    systems = ["R0", "R1", "A1", "A2"]
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    
    for idx, sys_name in enumerate(systems):
        wav_file = SAMPLES_DIR / f"{label.lower()}_{clip_id}_{sys_name}.wav"
        if not wav_file.exists():
            print(f"File not found: {wav_file}")
            continue
            
        wav, sr = sf.read(wav_file)
        
        # Compute log-mel spectrogram for visualization
        mel = librosa.feature.melspectrogram(
            y=wav, sr=sr, n_fft=1024, hop_length=256, win_length=1024, n_mels=80, fmin=0, fmax=8000
        )
        log_mel = librosa.power_to_db(mel, ref=np.max)
        
        librosa.display.specshow(
            log_mel, sr=sr, hop_length=256, x_axis="time", y_axis="mel", ax=axes[idx], cmap="magma"
        )
        axes[idx].set_title(f"{sys_name} - {label} ({clip_id})", fontsize=12, fontweight="bold")
        axes[idx].set_ylabel("Mel Band")
        
        print(f"System {sys_name}:")
        ipd.display(ipd.Audio(wav, rate=sr))
        
    plt.tight_layout()
    plt.show()

## 3. Best Performing Clip (`LJ005-0104`)
Transcript: *"The progress of improvement was certainly extremely slow."*
- **R0 WER:** 0.0%
- **R1 WER:** 12.5%
- **A1 WER:** 0.0%
- **A2 WER:** 0.0%

In [ ]:
compare_clip("LJ005-0104", "best")

## 4. Median Performing Clip (`LJ005-0293`)
Typical performance reflecting average intelligibility across the held-out test split.

In [ ]:
compare_clip("LJ005-0293", "median")

## 5. Worst Performing Clip (`LJ005-0006`)
Transcript: Long compound sentence with multiple clauses and rapid unstressed syllables.
- Visualizes how static templates blur transitions and struggle with rapid phonetic shifts.

In [ ]:
compare_clip("LJ005-0006", "worst")

## 6. Failure Analysis Log
Top worst-performing clips categorized by diagnosed root causes (*slurred transition, flat prosody, wrong duration, vocoder artefact*).

In [ ]:
failure_df = pd.read_csv(RESULTS_DIR / "failure_log.csv")
failure_df[["system", "rank", "clip_id", "wer", "cer", "guessed_cause"]].head(20)